In [ ]:
import os
import math
from collections import defaultdict, Counter
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.font_manager import FontProperties
from matplotlib.patches import ConnectionPatch, Rectangle
from matplotlib.ticker import MultipleLocator
import pandas as pd
import baltic as bt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
from matplotlib import transforms
import matplotlib.ticker as mticker
import seaborn as sns

In [ ]:
outdir = '../results/figures'

In [ ]:
def get_apobec_dimer(snp, dimer):
    if snp == 'G->A' and dimer == 'GA':
        return 'GA->AA'
    if snp == 'C->T' and dimer == 'TC':
        return 'TC->TT'
    return None


def count_apobec(dimers):
    n_apobec = sum([1 for d in dimers if d is not None])
    return n_apobec, len(dimers) - n_apobec


def load_apobec_counts_squirrel(fname):
    snps = defaultdict(list)
    with open(fname) as f_in:
        next(f_in)  # skip the header
        for line in f_in:
            parent, child, site, snp, dimer = map(str.strip, line.split(','))
            snps[(parent, child)].append(get_apobec_dimer(snp, dimer))
    apobec_counts = {
        (parent, child): count_apobec(dimers)
        for (parent, child), dimers in snps.items()
    }
    return apobec_counts


def load_apobec_counts_iqtree(fname):
    df_apobec_counts = pd.read_csv(fname)
    return {
        (row['parent'], row['child']): (row['apobec'], row['non-apobec'])
        for _, row in df_apobec_counts.iterrows()
    }


def load_apobec_counts(fname):
    with open(fname) as f:
        head = next(f).strip()
    if head.strip() == 'parent,child,apobec,non-apobec,unclear':
        return load_apobec_counts_iqtree(fname)
    return load_apobec_counts_squirrel(fname)


def plot_tips(ax, tree, marker, size, colour, markeredgecolor='black', markeredgewidth=0.3, do_plot=None, z_order=None, x_attr=None, y_attr=None):
    if x_attr is None:
        x_attr = lambda k: k.x
    if y_attr is None:
        y_attr = lambda k: k.y
    if z_order is None:
        z_order = lambda k: 10
    elif isinstance(z_order, int):
        z_order = lambda k: z_order
    if do_plot is None:
        do_plot = lambda k: True
    for l in tree.getBranches():
        if l.is_leaf():
            if do_plot(l):
                ax.plot(
                    [x_attr(l)], [y_attr(l)],
                    linestyle='',
                    marker=marker(l),
                    markersize=size(l),
                    color=colour(l),
                    zorder=z_order(l),
                    markeredgecolor=markeredgecolor,
                    markeredgewidth=markeredgewidth
                )


def plot_apobec(
        ax,
        tree,
        apobec_counts,
        apobec_marker='o',
        apobec_color='red',
        not_apobec_color='blue',
        marker_size=1,
        markeredgewidth=0.5,
        space_x=0.000005,
        space_y=0.5,
        shift_x=0.000006,
):
    inner_nodes = [
        node
        for node in tree.getBranches()
        if not node.is_leaf()
    ]
    for node in inner_nodes:
        node_id = node.traits['label']
        for child in node.children:
            child_id = (
                child.name
                if child.is_leaf()
                else child.traits['label']
            )
            n_apo, n_not = apobec_counts.get((node_id, child_id), (0, 0))
            base_x = node.x + shift_x
            base_y = child.y + space_y
            
            not_x = base_x + np.arange(n_not) * space_x
            not_y = np.array([base_y] * n_not)
            ax.plot(
                not_x, not_y,
                linestyle='',
                marker=apobec_marker,
                markersize=marker_size,
                color=not_apobec_color,
                markeredgewidth=markeredgewidth,
            )
            apo_x = base_x + (np.arange(n_apo) + n_not) * space_x
            apo_y = np.array([base_y] * n_apo)
            ax.plot(
                apo_x, apo_y,
                linestyle='',
                marker=apobec_marker,
                markersize=marker_size,
                color=apobec_color,
                markeredgewidth=markeredgewidth,
            )

def annotate_tree(tree, dataframe, acc_col='Accession'):
    annotations = {
        row[acc_col]: {
            k: ('NA' if str(v).lower() == 'nan' else v)
            for k, v in row.items()
            if k != acc_col
        }
        for _, row in dataframe.iterrows()
    }
    for leaf in tree.getBranches():
        if leaf.is_leaf():
            leaf.traits |= annotations.get(leaf.name, {})


def set_trait(node, trait):
    if node.is_leaf():
        node.traits.setdefault(trait, 'Unknown')
    if trait in node.traits:
        return node.traits[trait]
    traits = {
        set_trait(child, trait)
        for child in node.children
    }
    if len(traits) > 1:
        node.traits[trait] = 'Multi'
    else:
        node.traits[trait] = list(traits)[0]
    return node.traits[trait]


def label_by_common_feature(tree, trait='Country'):
    set_trait(tree.root, trait)


def add_legend(
        ax,
        mapping,
        title,
        feature='color',
        color='black',
        linestyle='',
        marker='o',
        title_fontsize=12,
        line_params=None,
        markeredgecolor='black',
        markeredgewidth=0.3,
        **legend_kwargs
):
    line_params = line_params if line_params else {}
    defaults = {
        'color': color,
        'linestyle': linestyle,
        'marker': marker,
        'markeredgecolor': markeredgecolor,
        'markeredgewidth': markeredgewidth
    } | line_params
    handles = []
    for label, feat in mapping.items():
        params = defaults | {feature: feat, 'label': label}
        params['markerfacecolor'] = params['color']
        handles.append(
            Line2D(
                [0], [0],
                **params
            )
        )
    kwargs = {
        'handles': handles,
        'title': title,
        'frameon': False,
        'title_fontproperties': FontProperties(weight='bold', size=title_fontsize),
        'alignment': 'left',
    } | legend_kwargs
    legend = ax.legend(**kwargs)
    ax.add_artist(legend)
    return legend


def round_first_nonzero(x):
    if x == 0:
        return 0
    digits = -math.floor(math.log10(abs(x)))
    return round(x, digits)


def add_scale(ax, tree, scale_factor, x, y, y_shift=None):
    if y_shift is None:
        y_shift = 1
    scale_len = round_first_nonzero(tree.treeHeight / scale_factor)
    ax.plot([x, x + scale_len], [y, y], lw=1, color='black')
    ax.text(
        x + scale_len / 2,
        y - y_shift,
        f'{scale_len}',
        ha='center',
        va='top'
    )

def trait_counts(tree, trait):
    return Counter([
        k.traits[trait]
        for k in tree.getBranches()
        if k.is_leaf()
    ])


def trait_vals(tree, trait):
    return sorted(set(trait_counts(tree, trait)))


In [ ]:
continent_to_country = {
    'Europe': [
        'Albania',
        'Andorra',
        'Armenia',
        'Austria',
        'Azerbaijan',
        'Belarus',
        'Belgium',
        'Bosnia and Herzegovina',
        'Bulgaria',
        'Croatia',
        'Cyprus',
        'Czechia',
        'Denmark',
        'Estonia',
        'Finland',
        'France',
        'Georgia',
        'Germany',
        'Greece',
        'Hungary',
        'Iceland',
        'Ireland',
        'Italy',
        'Kosovo',
        'Latvia',
        'Liechtenstein',
        'Lithuania',
        'Luxembourg',
        'Malta',
        'Moldova',
        'Monaco',
        'Montenegro',
        'Netherlands',
        'North Macedonia',
        'Norway',
        'Poland',
        'Portugal',
        'Romania',
        'Russia',
        'San Marino',
        'Serbia',
        'Slovakia',
        'Slovenia',
        'Spain',
        'Sweden',
        'Switzerland',
        'Turkey',
        'Ukraine',
        'United Kingdom',
        'Vatican City'
    ]
}
country_to_continent = {
    country: continent
    for continent, countries in continent_to_country.items()
    for country in countries
}
country_fix = {
    "Cote d'Ivoire": "Côte d'Ivoire",
    **country_to_continent
}

def add_corrected_country(tree):
    for leaf in tree.getExternal():
        country = leaf.traits['Country']
        leaf.traits['CountryCorrected'] = country_fix.get(country, country)

In [ ]:
def load_and_annotate_tree(fname_tree, fname_snps=None, fname_meta=None, acc_col='Accession'):
    df_meta = pd.read_csv(fname_meta, sep='\t').rename(columns={'geoLocCountry': 'Country'})
    try:
        tree = bt.loadNexus(fname_tree, absoluteTime=False)
    except:
        tree = bt.loadNewick(fname_tree, absoluteTime=False)
    tree.traverse_tree()
    if fname_snps is not None:
        apobec_counts = load_apobec_counts(fname_snps)
    else:
        apobec_counts = None
    annotate_tree(tree, df_meta, acc_col)
    label_by_common_feature(tree, trait='Country')
    add_corrected_country(tree)
    return tree, apobec_counts

In [ ]:
fname_meta_cladeIIa = '../data/IIa/meta_all.tsv'
tree_IIa, apobec_IIa = load_and_annotate_tree(
    fname_tree='../results/IIa/iqtree/model_comparison.4baltic.treefile',
    fname_snps='../results/IIa/iqtree/model_comparison.apobec_counts.csv',
    fname_meta=fname_meta_cladeIIa,
)

tree_IIb, apobec_IIb = load_and_annotate_tree(
    fname_tree=f'../results/IIb/iqtree/model_comparison.4baltic.treefile',
    fname_snps=f'../results/IIb/iqtree/model_comparison.apobec_counts.csv',
    fname_meta=f'../data/IIb/meta.tsv',
    acc_col='accessionVersion',
)

In [ ]:
countries_IIa = trait_counts(tree_IIa, 'CountryCorrected')
countries_IIa

In [ ]:
countries_IIb = trait_counts(tree_IIb, 'CountryCorrected')
countries_IIb

In [ ]:
color_table = sns.color_palette('colorblind', 10)

n_countries_to_color = len(color_table) - 1
countries_to_color = (
    [c for c, _ in countries_IIa.most_common()][:n_countries_to_color]
    + [c for c, _ in countries_IIb.most_common() if c not in countries_IIa][:max(0, n_countries_to_color - len(countries_IIa))]
)
countries_to_color

In [ ]:
country_to_color = {
    'Benin': color_table[5],
    "Côte d'Ivoire": color_table[4],
    'Europe': color_table[0],
    'Ghana':  color_table[2],
    'Guinea': color_table[3],
    'Liberia': color_table[8],
    'Nigeria': color_table[6],    
    'Sierra Leone': color_table[9],
    'USA': color_table[1],
    'Other': color_table[7],
}

host_to_marker = {
    'Human': 'o',
    'Non-human': '^',
    'Unknown': 's'
}

apo_descr = 'APOBEC3 signature'
other_descr = 'Other'

apobec_colors = {
    apo_descr: 'black',
    other_descr: 'orange',
}
apobec_marker = 'D'

color_func_country = lambda k: country_to_color.get(k.traits.get('CountryCorrected', 'Other'), country_to_color['Other'])
size_func_marker = lambda k: (
    10
    if k.name.startswith('G1')
    else 6
)
marker_func_host = lambda k: host_to_marker.get(k.traits.get('Host', 'Unknown'), host_to_marker['Non-human'])

In [ ]:
map_extent = [-15.5, -7.0, 6.0, 13.5]
bbox = box(*map_extent)

shp = shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shp)

country_names_cartopy = set()

for rec in reader.records():
    if rec.geometry.intersects(bbox):
        # different files sometimes use NAME or NAME_LONG
        name = rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME')
        country_names_cartopy.add(name)

In [ ]:
map_extent = [-15.5, -7.0, 6.0, 13.5]
bbox = box(*map_extent)

shp = shpreader.natural_earth(resolution='10m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shp)

country_names_cartopy = set()

for rec in reader.records():
    if rec.geometry.intersects(bbox):
        # different files sometimes use NAME or NAME_LONG
        name = rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME')
        country_names_cartopy.add(name)

for n in sorted(country_names_cartopy):
    print(n)

country_name_map = {
    #"Côte d'Ivoire" : "Cote d'Ivoire"
}

country_names_cartopy_remapped = {country_name_map.get(c, c) for c in country_names_cartopy}

print(country_names_cartopy_remapped.intersection(country_to_color))
print(set(country_to_color) - country_names_cartopy_remapped)
print(country_names_cartopy_remapped - set(country_to_color))

In [ ]:
geo_dat = pd.read_csv('../data/geodata.tsv', sep='\t')
sourcetable = pd.read_csv('../data/sourcetable.tsv', sep='\t')

goloc_col = 'Country (geo_loc_name)'
sourcetable[goloc_col] = sourcetable[goloc_col]

sourcetab_longlat = pd.merge(sourcetable, geo_dat, left_on=goloc_col, right_on='place').set_index('Sequence_ID')

In [ ]:
sourcetable

In [ ]:
sourcetab_longlat

In [ ]:
def contains_guinea_leaves(node):
    if node.is_leaf():
        return node.name.startswith('G1')
    return any(l.startswith('G1') for l in node.leaves)


def set_avg_lat(node, df_longlat):
    if node.is_leaf():
        node.traits['average_latitude'] = df_longlat.loc[node.name, 'latitude']
    else:
        gin_children = list(filter(contains_guinea_leaves, node.children))
        for child in gin_children:
            set_avg_lat(child, sourcetab_longlat)
        node.traits['average_latitude'] = np.mean([child.traits['average_latitude'] for child in gin_children])


def untangle_geography(tree, df_longlat):
    subtree = None
    # find the subtree containing all giunea leaves in IIa
    to_process = [tree.root]
    while to_process:
        node = to_process.pop()
        gin_children = [
            child
            for child in node.children
            if contains_guinea_leaves(child)
        ]
        if len(gin_children) > 1:
            subtree = node
        else:
            to_process.extend([child for child in gin_children if not child.is_leaf()])
    if subtree is None:
        raise ValueError('Could not find a subtree containing all Guinea leaves')
    set_avg_lat(subtree, df_longlat)
    
    to_process = [subtree]
    while to_process:
        node = to_process.pop()
        node.children = sorted(node.children, key=lambda c: c.traits.get('average_latitude', 0), reverse=True)
        to_process.extend([child for child in node.children if not child.is_leaf()])
    tree.drawTree()


untangle_geography(tree_IIa, sourcetab_longlat)

In [ ]:
size_func_marker_mapplot_IIb = lambda k: (
    8
    if k.name.startswith('G1')
    else 3
)
size_func_marker_mapplot_IIa = lambda k: (
    8
    if k.name.startswith('G1')
    else 6
)

# create the figure
fig = plt.figure(figsize=(12, 4))

ax_IIb = fig.add_axes([0.01, 0.01, 0.294, 0.98])
ax_map = fig.add_axes(
    [0.314, 0.01, 0.392, 0.98],
    projection=ccrs.PlateCarree()
)
ax_map.set_aspect('equal', adjustable='box')
ax_IIa = fig.add_axes([0.716, 0.01, 0.294, 0.98])

# Tree clade IIb

title_params = {
    'fontsize': 15,
    'pad': 8,
    'loc': 'left',
    'fontweight': 'bold',
}

ax_IIb.set_title(r'B) Clade IIb lineage A.2', **title_params)

ax_IIb.set_xlim(-0.01 * tree_IIb.treeHeight, 1.1 * tree_IIb.treeHeight)
ax_IIb.set_ylim(-0.01 * tree_IIb.ySpan, 1.01 * tree_IIb.ySpan)
ax_IIb.axis('off')

tree_IIb.plotTree(
    ax_IIb,
    colour='black',
    width=1
)

plot_tips(
    ax_IIb,
    tree_IIb,
    marker=lambda k: 'o',
    size=size_func_marker_mapplot_IIb,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') else 5
)

add_legend(
    ax_IIb,
    country_to_color,
    'Country',
    loc='lower left',
    bbox_to_anchor=(0, 0),
    marker='o',
    linestyle='',
    fontsize=11,
    title_fontsize=12,
)

add_scale(ax_IIb, tree_IIb, 10, tree_IIb.treeHeight / 2, tree_IIb.ySpan / 2, y_shift=tree_IIb.ySpan / 100)

# add clade marker
lin_G1_leaves = [
    leaf
    for leaf in tree_IIb.getBranches()
    if leaf.is_leaf() and leaf.traits['lineage'] == 'G.1'
]
min_y_G1 = min([l.y for l in lin_G1_leaves])
max_y_G1 = max([l.y for l in lin_G1_leaves])
x_G1 = 1.025 * tree_IIb.treeHeight

ax_IIb.annotate(
    '',
    xy=(x_G1, (max_y_G1 - min_y_G1) * 9 / 10),      # arrow tip (bottom)
    xytext=(x_G1, max_y_G1),  # arrow start (top)
    arrowprops=dict(
        arrowstyle='-|>',
        color='grey',
        linewidth=2,
        shrinkA=0,
        shrinkB=0,
    ),
)
ax_IIb.text(
    x_G1 * 1.01,
    max_y_G1,
    'G.1',
    color='black',
    fontsize=11,
    ha='left',   # text to the right of the line
    va='top',    # top aligned with line top
)

# MAP
ax_map.add_feature(cfeature.LAND)
ax_map.add_feature(cfeature.OCEAN)
ax_map.add_feature(cfeature.BORDERS, linewidth=0.5)
ax_map.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax_map.set_extent(map_extent, crs=ccrs.PlateCarree())

# latitude and longitude labels
gl = ax_map.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    x_inline=False,
    y_inline=False,
)

gl.top_labels = False
gl.left_labels = False
gl.right_labels = True

# hide gridlines, show only coordinate labels
gl.xlines = False
gl.ylines = False

# label formatting
gl.xlabel_style = {'size': 9}
gl.ylabel_style = {'size': 9}

gl.xlocator = mticker.MultipleLocator(2)
gl.ylocator = mticker.MultipleLocator(2)

# draw countries with per-country colors
for rec in reader.records():
    country_rec = rec.attributes.get('NAME_LONG') or rec.attributes.get('NAME')  # depends on dataset version
    country_name = country_name_map.get(country_rec, country_rec)
    
    if country_name in country_to_color:
        color = country_to_color[country_name]
        ax_map.add_geometries(
            [rec.geometry],
            crs=ccrs.PlateCarree(),
            facecolor=color,
            edgecolor='black',
            linewidth=0.3,
            zorder=2
        )

# clade IIa phylogeny
invert_x = lambda x: None if x is None else -x
invert_x_node = lambda k: invert_x(k.x)

ax_IIa.axis('off')
tree_IIa.plotTree(
    ax_IIa,
    colour='black',
    width=1,
    x_attr=invert_x_node,
)
plot_tips(
    ax_IIa,
    tree_IIa,
    marker=marker_func_host,
    size=size_func_marker_mapplot_IIa,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') else 5,
    x_attr=invert_x_node,
)

add_legend(
    ax_IIa,
    host_to_marker,
    'Host',
    feature='marker',
    loc='lower left',
    bbox_to_anchor=(0.6, 0),
    linestyle='',
    color='grey',
    fontsize=11,
    title_fontsize=12,
)
add_scale(ax_IIa, tree_IIa, 10, -tree_IIa.treeHeight / 2, tree_IIa.ySpan / 2, y_shift=tree_IIa.ySpan / 100)

ax_IIa.set_title(r'A) Clade IIa', **title_params)
ax_IIa.set_xlim(-1.1 * tree_IIa.treeHeight, 0.01 * tree_IIa.treeHeight)
ax_IIa.set_ylim(-0.01 * tree_IIa.ySpan, 1.01 * tree_IIa.ySpan)

# plot lines connecting trees and places
connect_color='black'
connect_alpha=0.4
connect_linewidth=1.5
connect_zorder=1

linestyles = [
    '-',                         # solid
    '--',                        # dashed
    '-.',                        # dash-dot
    ':',                         # dotted
    (0, (5, 1)),                 # long dash, tight gap
    (0, (3, 1, 1, 1)),           # dash-dot-dot
    (0, (1, 1)),                 # dense dots
    (0, (5, 2, 1, 2)),           # long dash + dot
    (0, (3, 2, 3, 2, 1, 2)),     # dash dash dot
    (0, (8, 2)),                 # very long dash
]

ll_to_map = ccrs.PlateCarree()._as_mpl_transform(ax_map)

guinea_leaves_IIb = [
    (leaf, True)
    for leaf in tree_IIb.getBranches()
    if leaf.is_leaf() and leaf.name.startswith('G1')
]
guinea_leaves_IIa = [
    (leaf, False)
    for leaf in tree_IIa.getExternal(lambda k: k.name.startswith('G1'))
    if leaf.is_leaf() and leaf.name.startswith('G1')
]
guinea_leaves = guinea_leaves_IIb + guinea_leaves_IIa

for i, (leaf, forward) in enumerate(guinea_leaves):
    lat = sourcetab_longlat.loc[leaf.name, 'latitude']
    long = sourcetab_longlat.loc[leaf.name, 'longitude']

    y_leaf = leaf.y
    
    connect_linestyle = linestyles[i % len(linestyles)]

    if forward:
        ax_tree = ax_IIb
        x_connnect_point = tree_IIb.treeHeight * 1.1
        x_leaf = leaf.x * 1.05
    else:
        ax_tree = ax_IIa
        x_connnect_point = -tree_IIa.treeHeight * 1.1
        x_leaf = -leaf.x * 1.05
    
    x_connect = x_connnect_point

    ax_tree.plot(
        [x_leaf, x_connnect_point],
        [y_leaf, y_leaf],
        linestyle=connect_linestyle,
        color=connect_color,
        alpha=connect_alpha,
        linewidth=connect_linewidth,
        zorder=connect_zorder,
    )

    con = ConnectionPatch(
        xyA=(x_connnect_point, y_leaf),
        coordsA=ax_tree.transData,
        xyB=(long, lat),
        coordsB=ll_to_map,
        #arrowstyle='-',
        linestyle=connect_linestyle,
        linewidth=connect_linewidth,
        alpha=connect_alpha,
        zorder=connect_zorder,
        clip_on=False,
        color=connect_color,
    )
    fig.add_artist(con)

    ax_map.plot(
        long,
        lat,
        marker='o',
        markersize=3.5,
        markeredgewidth=0.4,
        markeredgecolor='black',
        color='white',
        transform=ccrs.PlateCarree(),
        zorder=5,
    )

In [ ]:
os.makedirs(outdir, exist_ok=True)
fig.savefig(f'{outdir}/trees_and_countries.pdf', bbox_inches='tight')

## Maximum Likelihood plots with outgroups

In [ ]:
fname_tree_cladeIIa_outgroup = '../results/IIa/iqtree/model_comparison.4baltic.with_outgroup.treefile'

case='linA2'
fname_tree_cladeIIb_outgroup = f'../results/IIb/iqtree/model_comparison.4baltic.with_outgroup.treefile'

tree_IIa_outgroup = load_and_annotate_tree(
    fname_tree=fname_tree_cladeIIa_outgroup,
    fname_meta=fname_meta_cladeIIa,
)[0]

tree_IIb_outgroup = load_and_annotate_tree(
    fname_tree=fname_tree_cladeIIb_outgroup,
    fname_meta=f'../data/IIb/meta.tsv',
    acc_col='accessionVersion',
)[0]

In [ ]:
fig, (ax_IIa, ax_IIb) = plt.subplots(1, 2, figsize=(10, 10))

title_params = {
    'fontsize': 15,
    'pad': 8,
    'loc': 'left',
    'fontweight': 'bold',
}


ax_IIa.set_title(r'A) Clade IIa', **title_params)
ax_IIb.set_title(r'B) Clade IIb lineage A.2', **title_params)

ax_IIa.axis('off')
ax_IIb.axis('off')

# Clade IIa with outgroup

def sizefunc_rooting_IIa(k):
    if k.name.startswith('G1'):
        return 8
    elif k.name == outgroup_IIa:
        return 10
    return 6

outgroup_IIa = 'NC_063383.1'
outgroup_IIa_seqid = 'NC_063383'

IIa_outgroup_y_attr = lambda k: k.y * 1.1 if k.is_leaf() and k.name == outgroup_IIa else k.y

tree_IIa_outgroup.plotTree(
    ax_IIa,
    colour='black',
    width=1,
    y_attr=IIa_outgroup_y_attr,
)

tree_IIa_outgroup.addText(
    ax_IIa,
    text=lambda k: f'Outgroup ({outgroup_IIa_seqid})',
    target=lambda k: k.is_leaf() and k.name == outgroup_IIa,
    y_attr=IIa_outgroup_y_attr,
    x_attr=lambda k: k.x * 1.1 if k.is_leaf() and k.name == outgroup_IIa else k.x
)

plot_tips(
    ax_IIa,
    tree_IIa_outgroup,
    marker=lambda k: 'o' if k.name == outgroup_IIa else marker_func_host(k),
    size=sizefunc_rooting_IIa,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') or k.name == outgroup_IIa else 5,
    y_attr=IIa_outgroup_y_attr,
)

add_legend(
    ax_IIa,
    host_to_marker,
    'Host',
    feature='marker',
    loc='lower left',
    bbox_to_anchor=(0, 0),
    linestyle='',
    color='grey',
    fontsize=11,
    title_fontsize=12,
)
add_scale(ax_IIa, tree_IIa_outgroup, 10, tree_IIa_outgroup.treeHeight / 2, tree_IIa_outgroup.ySpan / 2, y_shift=tree_IIa_outgroup.ySpan / 100)


# Clade IIb with outgroup

def sizefunc_rooting_IIb(k):
    if k.name.startswith('G1'):
        return 8
    elif k.name == outgroup_IIb:
        return 10
    return 3

outgroup_IIb = 'KJ642615|human|Nigeria||1978'
outgroup_IIb_seqid = 'KJ642615'

IIb_outgroup_y_attr = lambda k: k.y * 1.1 if k.is_leaf() and k.name == outgroup_IIb else k.y

tree_IIb_outgroup.plotTree(
    ax_IIb,
    colour='black',
    width=1,
    y_attr=IIb_outgroup_y_attr
)

tree_IIb_outgroup.addText(
    ax_IIb,
    text=lambda k: f'Outgroup ({outgroup_IIb_seqid})',
    target=lambda k: k.is_leaf() and k.name == outgroup_IIb,
    y_attr=IIb_outgroup_y_attr,
    x_attr=lambda k: k.x * 1.1 if k.is_leaf() and k.name == outgroup_IIb else k.x
)

plot_tips(
    ax_IIb,
    tree_IIb_outgroup,
    marker=lambda k: 'o',
    size=sizefunc_rooting_IIb,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') or k.name == outgroup_IIb else 5,
    y_attr=IIb_outgroup_y_attr,
)

add_legend(
    ax_IIb,
    country_to_color,
    'Country',
    loc='lower left',
    bbox_to_anchor=(0, 0),
    marker='o',
    linestyle='',
    fontsize=11,
    title_fontsize=12,
)

add_scale(ax_IIb, tree_IIb_outgroup, 10, tree_IIb_outgroup.treeHeight / 2, tree_IIb_outgroup.ySpan / 2, y_shift=tree_IIb_outgroup.ySpan / 100)

In [ ]:
fig.savefig(f'{outdir}/outgroups.pdf', bbox_inches='tight')

## Clade IIa maximum likelihood (with APOBEC3 variants)

In [ ]:
country_to_color_IIa = {
    country: color
    for country, color in country_to_color.items()
    if country in countries_IIa
}

In [ ]:
def apobec_plot_IIa(ax, tree, apobec_counts, country_colors, show_bootstrap=False):

    ax.axis('off')
    tree.plotTree(
        ax,
        colour='black',
        width=1,
    )
    if show_bootstrap:
        tree.addText(
            ax,
            target=lambda k: 'bootstrap' in k.traits,
            text=lambda k: f'{k.traits["bootstrap"]}'
        )
    plot_tips(
        ax,
        tree,
        marker=marker_func_host,
        size=lambda k: 5 if k.name.startswith('G1') else 4,
        colour=color_func_country,
        z_order=lambda k: 10 if k.name.startswith('G1') else 5,
    )
    plot_apobec(
        ax,
        tree,
        apobec_counts,
        apobec_marker=apobec_marker,
        apobec_color=apobec_colors[apo_descr],
        not_apobec_color=apobec_colors[other_descr],
        marker_size=2,
        space_x=0.000004,
        space_y=0.5,
        shift_x=0.000003,
    )

    add_legend(
        ax,
        country_colors,
        'Country',
        loc='lower left',
        bbox_to_anchor=(0, 0),
        marker='o',
        linestyle='',
        fontsize=9,
        title_fontsize=10,
    )
    add_legend(
        ax,
        host_to_marker,
        'Host',
        feature='marker',
        loc='lower left',
        bbox_to_anchor=(0.26, 0.2),
        linestyle='',
        color='grey',
        fontsize=9,
        title_fontsize=10,
    )
    add_legend(
        ax,
        apobec_colors,
        'SNP',
        loc='lower left',
        bbox_to_anchor=(0.26, 0),
        marker=apobec_marker,
        linestyle='',
        fontsize=9,
        title_fontsize=10,
        markeredgewidth=0.0,
        markeredgecolor='none'
    )
    add_scale(ax, tree, 10, tree.treeHeight / 2, tree.ySpan / 2)

In [ ]:
fig_apobec, ax_apobec = plt.subplots(1, 1)
apobec_plot_IIa(ax_apobec, tree_IIa, apobec_IIa, country_to_color_IIa)

In [ ]:
fig_apobec.savefig(f'{outdir}/cladeIIa_apobec.pdf', bbox_inches='tight')

## Clade IIb maximum likelihood phylogeny with APOBEC3 variants and bootstrap values

In [ ]:
def bootstrap_color(node):
    if not 'bootstrap' in node.traits:
        return 'black'
    bs = int(node.traits['bootstrap'])
    if bs > 90:
        return 'green'
    elif bs > 70:
        return 'orange'
    return 'red'

In [ ]:
fig_IIb_zoom, ax = plt.subplots(1, 1, figsize=(10, 14))

ax.axis('off')

tree_IIb.plotTree(
    ax,
    colour=bootstrap_color,
    width=1
)
tree_IIb.addText(
    ax,
    target=lambda k: not k.is_leaf() and k.traits['label'] == 'Node59',
    text=lambda k: k.traits['label']
)

plot_tips(
    ax,
    tree_IIb,
    marker=lambda k: 'o',
    size=lambda k: 5 if k.name.startswith('G1') else 2,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') else 5
)

add_legend(
    ax,
    country_to_color,
    'Country',
    loc='lower left',
    bbox_to_anchor=(0, 0),
    marker='o',
    linestyle='',
    fontsize=11,
    title_fontsize=12,
)

add_scale(ax, tree_IIb, 10, tree_IIb.treeHeight / 2, tree_IIb.ySpan / 2)

In [ ]:
fig_cladeIIb_bootstrap, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12, 12))

ax_left.axis('off')
ax_right.axis('off')

tree_IIb.plotTree(
    ax_left,
    colour='black',
    width=1,
)

tree_IIb.plotPoints(
    ax_left,
    target=lambda k: not k.is_leaf() and int(k.traits.get('bootstrap', 0)) >= 90,
    size=50,
    colour='grey',
    outline=False,
    zorder=1
)

ax_left.set_xlim(-0.01 * tree_IIb.treeHeight, 1.02 * tree_IIb.treeHeight)
ax_left.set_ylim(-0.01 * tree_IIb.ySpan, 1.01 * tree_IIb.ySpan)

plot_tips(
    ax_left,
    tree_IIb,
    marker=lambda k: 'o',
    size=lambda k: 8 if k.name.startswith('G1') else 4,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') else 5
)

legend_fontsize = 11
legend_title_fontsize = 12
add_legend(
    ax_left,
    country_to_color,
    'Country',
    loc='lower left',
    bbox_to_anchor=(0, 0.23),
    marker='o',
    linestyle='',
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
)
add_legend(
    ax_left,
    {'≥ 90 %': 'o', '< 90 %': 'none'},
    'Bootstrap support',
    color='grey',
    feature='marker',
    loc='lower left',
    bbox_to_anchor=(0, 0.1),
    linestyle='',
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    line_params=dict(markersize=10),
    markeredgewidth=0.0,
    markeredgecolor='none'
)
add_legend(
    ax_left,
    apobec_colors,
    'SNP',
    loc='lower left',
    bbox_to_anchor=(0, 0.0),
    marker=apobec_marker,
    linestyle='',
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    markeredgewidth=0.0,
    markeredgecolor='none'
)


add_scale(ax_left, tree_IIb, 10, tree_IIb.treeHeight / 2, tree_IIb.ySpan / 2)

g1_leaves = tree_IIb.getExternal(lambda k: k.traits['lineage'] == 'G.1')

min_x = min([l.x for l in g1_leaves])
max_x = max([l.x for l in g1_leaves])
max_y = max([l.y for l in g1_leaves])
min_y = min([l.y for l in g1_leaves])

margin_factor_x = 0.05
margin_factor_y = 0.01
margin_x = (max_x - min_x) * margin_factor_x
margin_y = (max_y - min_y) * margin_factor_y

xmax = max_x + margin_x
xmin = min_x - margin_x
ymax = max_y + margin_y
ymin = min_y - margin_y

rect = Rectangle(
    (xmin, ymin),          # lower-left corner
    xmax - xmin,           # width
    ymax - ymin,           # height
    facecolor='none',
    edgecolor='grey',
    linewidth=2,
    alpha=1.0,
    zorder=-1               # behind tree
)
ax_left.add_patch(rect)

ax_left.text(
    xmax,
    ymax,
    'lineage G.1',
    color='grey',
    fontsize=11,
    ha='right',   # text to the right of the line
    va='bottom',    # top aligned with line top
)

# get the G1 subclade tree
node59 = tree_IIb.getBranches(lambda k: not k.is_leaf() and k.traits['label'] == 'Node59')
tips = tree_IIb.getExternal(lambda k: k.name in node59.leaves)
g1_subtree = tree_IIb.reduceTree(tips)
print(len(g1_subtree.getExternal()), len(g1_subtree.getInternal()))
g1_subtree.traverse_tree()
g1_subtree.plotTree(
    ax_right,
    colour='black',
    width=1,
)
plot_tips(
    ax_right,
    g1_subtree,
    marker=lambda k: 'o',
    size=lambda k: 8 if k.name.startswith('G1') else 4,
    colour=color_func_country,
    z_order=lambda k: 10 if k.name.startswith('G1') else 5
)

plot_apobec(
    ax_right,
    g1_subtree,
    apobec_IIb,
    apobec_marker=apobec_marker,
    apobec_color=apobec_colors[apo_descr],
    not_apobec_color=apobec_colors[other_descr],
    marker_size=3,
    space_x=0.000002,
    space_y=1.0,
    shift_x=0.000003,
)

ax_left.set_title('A) Clade IIb lineage A.2', fontsize=15, pad=8, loc='left', fontweight='bold')
ax_right.set_title('B) Lineage G.1', fontsize=15, pad=8, loc='left', fontweight='bold')

ax_right.set_xlim(0.99 * node59.x, 1.02 * g1_subtree.treeHeight)
ax_right.set_ylim(-0.01 * g1_subtree.ySpan, 1.01 * g1_subtree.ySpan)

In [ ]:
fig_cladeIIb_bootstrap.savefig(f'{outdir}/cladeIIb_bootstrap_and_apobec.pdf', bbox_inches='tight')

Compute the number of apobec variants in the G.1 subtree and for the Guinea genomes.

In [ ]:
g1_subtree.getExternal()[0].traits

n_apo_total = 0
n_not_total = 0

n_apo_gin = 0
n_not_gin = 0

to_process = [g1_subtree.root]
while to_process:
    node = to_process.pop()
    node_id = node.traits['label']
    for child in node.children:
        if child.is_leaf():
            child_id = child.name
        else:
            to_process.append(child)
            child_id = child.traits['label']
        n_apo, n_not = apobec_IIb.get((node_id, child_id), (0, 0))
        n_apo_total += n_apo
        if child_id.startswith('G1'):
            n_apo_gin += n_apo
            n_not_gin += n_not
        n_not_total += n_not

print(f"{n_apo_total} / {n_apo_total + n_not_total}")
print(f"{n_apo_gin} / {n_apo_gin + n_not_gin}")


## Time Trees (clade IIa)

In [ ]:
fname_timetree = '../results/IIa/beast/strict_clock/mcc_median.tree'
timetree_IIa = bt.loadNexus(fname_timetree, absoluteTime=False)

max_time = max(
    float(l.name.split('_')[-1])
    for l in timetree_IIa.getExternal()
)

for leaf in timetree_IIa.getExternal():
    leaf.name = leaf.name.split('_')[0]

timetree_IIa.traverse_tree()
timetree_IIa.setAbsoluteTime(max_time)

In [ ]:
df_meta = pd.read_csv(fname_meta_cladeIIa, sep='\t').rename(columns={'geoLocCountry': 'Country'})
df_meta['acc_corrected'] = df_meta['Accession'].apply(lambda x: x.replace('_', ''))
annotate_tree(timetree_IIa, df_meta, 'acc_corrected')
label_by_common_feature(timetree_IIa, trait='Country')
add_corrected_country(timetree_IIa)
label_by_common_feature(timetree_IIa, trait='CountryCorrected')


In [ ]:
fig_timetree, (ax_left, ax_mid) = plt.subplots(1, 2, figsize=(8, 4))

title_params_timetree = {
    'fontsize': 15,
    'pad': 8,
    'loc': 'left',
}

# Time trees

x_attr=lambda k: k.absoluteTime

min_x_zoom = 1985
max_x_zoom = 2027
min_x_zoom_box = 1995
max_x_zoom_box = 2027
min_y_zoom = min(k.y for k in timetree_IIa.getExternal(lambda k: k.name.startswith('G1')))
max_y_zoom = max(k.y for k in timetree_IIa.getExternal(lambda k: k.name.startswith('G1')))

timetree_IIa.plotTree(
    ax_left,
    x_attr=x_attr,
    colour='black',
    width=1
)

plot_tips(
    ax_left,
    timetree_IIa,
    marker=marker_func_host,
    size=lambda k: 4 if k.name.startswith('G1') else 3,
    colour=color_func_country,
    x_attr=x_attr,
)

add_legend(
    ax_left,
    country_to_color_IIa,
    'Country',
    loc='lower left',
    bbox_to_anchor=(0, 0),
    fontsize=7,
    title_fontsize=8,
)
add_legend(
    ax_left,
    host_to_marker,
    'Host',
    feature='marker',
    loc='lower left',
    bbox_to_anchor=(0.35, 0.0),
    linestyle='',
    color='grey',
    fontsize=7,
    title_fontsize=8,
)

min_y = min(k.y for k in timetree_IIa.getExternal())
max_y = max(k.y for k in timetree_IIa.getExternal())

margin = 0.05 * (max_y - min_y)
ymin = min_y - margin
ymax = max_y + margin

min_x = 1800
max_x = 2035
step_size = 25

ax_left.set_xlabel('Year')

ax_left.xaxis.set_major_locator(MultipleLocator(step_size))
ax_left.tick_params(axis='x', labelrotation=45, labelsize=8) 

ax_left.set_xlim(min_x, max_x)
ax_left.set_ylim(-1, timetree_IIa.ySpan + 1)

ax_left.yaxis.set_visible(False)

ax_left.spines['top'].set_visible(False)
ax_left.spines['right'].set_visible(False)
ax_left.spines['left'].set_visible(False)

# add box for zoom region

margin_factor_x = 0.2
margin_factor_y = 0.2
margin_x = (max_x_zoom_box - min_x_zoom_box) * margin_factor_x
margin_y = (max_y_zoom - min_y_zoom) * margin_factor_y

xmax = max_x_zoom_box + margin_x
xmin = min_x_zoom_box - margin_x
ymax = max_y_zoom + margin_y
ymin = min_y_zoom - margin_y

rect = Rectangle(
    (xmin, ymin),          # lower-left corner
    xmax - xmin,           # width
    ymax - ymin,           # height
    facecolor='none',
    edgecolor='grey',
    linewidth=2,
    alpha=1.0,
    zorder=-1               # behind tree
)
print(rect)
ax_left.add_patch(rect)

ax_left.text(
    xmin,
    ymax,
    'B',
    color='grey',
    fontsize=11,
    ha='left',
    va='bottom',
)

ax_left.text(
    min_x,
    timetree_IIa.ySpan + 1,
    'A',
    fontweight='bold',
    color='black',
    fontsize=15,
    ha='left',
    va='top',
)


## Zoom

min_y = min_y_zoom
max_y = max_y_zoom

margin = 0.05 * (max_y - min_y)
ymin = min_y - margin
ymax = max_y + margin

timetree_IIa.plotTree(
    ax_mid,
    x_attr=x_attr,
    colour='black',
    width=1
)
timetree_IIa.addText(
    ax_mid,
    text=lambda k: f"{k.traits.get('posterior', ''):.2f}",
    target=lambda k: k.index in [5,6,7,363,364,2590],
    x_attr=x_attr,
    size=7,
)
plot_tips(
    ax_mid,
    timetree_IIa,
    marker=marker_func_host,
    size=lambda k: 10 if k.name.startswith('G1') else 5,
    colour=color_func_country,
    x_attr=x_attr,
)

# plot bars
for node in timetree_IIa.getInternal():
    x = (max_time - np.array(node.traits['height_95%_HPD']))[::-1]
    ax_mid.plot(
        x, [node.y, node.y],
        linestyle='-',
        #marker='|',
        marker=None,
        color='lightblue',
        zorder=0,
        linewidth=2.5,
        alpha=1.0
    )

step_size = 5

for x in range(min_x_zoom, max_x_zoom, step_size * 2):
    # make background
    rect = Rectangle(
        (x, ymin),          # lower-left corner
        step_size,           # width
        ymax - ymin,           # height
        facecolor='grey',
        edgecolor='none',
        linewidth=0,
        alpha=0.2,
        zorder=-1               # behind tree
    )
    ax_mid.add_patch(rect)

ax_mid.set_xlabel('Year')
ax_mid.set_xlim(min_x_zoom, max_x_zoom)

ax_mid.xaxis.set_major_locator(MultipleLocator(step_size))
ax_mid.tick_params(axis='x', labelrotation=45, labelsize=8) 

ax_mid.set_ylim(ymin, ymax)

ax_mid.yaxis.set_visible(False)
ax_mid.spines['top'].set_visible(False)
ax_mid.spines['right'].set_visible(False)
ax_mid.spines['left'].set_visible(False)

ax_mid.text(
    min_x_zoom,
    ymax,
    'B',
    fontweight='bold',
    color='black',
    fontsize=15,
    ha='left',   # text to the right of the line
    va='top',    # top aligned with line top
)

In [ ]:
fig_timetree.savefig(f'{outdir}/timetree_side_by_side.pdf', bbox_inches='tight')

In [ ]:
ancestors = []
for node in timetree_IIa.getBranches():
    if not node.is_leaf():
        gin = len([
            l
            for l in node.leaves
            if l.startswith('G1')
        ])
        not_gin = len(node.leaves) - gin
        if gin > 0 and not_gin <= 2:
            up, lo = max_time - np.array(node.traits['height_95%_HPD'])
            ancestors.append({
                'leaves': ' '.join(sorted(node.leaves)),
                'date': int(round(node.absoluteTime, 0)),
                '95%_low': int(round(lo, 0)),
                '95%_up': int(round(up, 0))
            })

df_anc = pd.DataFrame(ancestors)

print(df_anc)

df_anc.to_csv(f'{outdir}/common_ancestor_times.tsv', sep='\t')
